In [1]:
# SimpleDirectoryReader is dynamic, detects file type and uses appropriate reader
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings, PromptTemplate
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_parse import LlamaParse

from transformers import AutoTokenizer
from transformers import pipeline as hf_pipeline
from sentence_transformers import SentenceTransformer, util

import pandas as pd, re, ast, textwrap
from datasets import Dataset

from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.metrics import answer_relevancy as m_answer_relevancy
from ragas.metrics import faithfulness as m_faithfulness
from ragas.metrics import context_recall as m_context_recall
from ragas.metrics import context_precision as m_context_precision
from ragas import evaluate

from transformers import pipeline as hf_pipeline

from dotenv import load_dotenv, find_dotenv

import torch
import os
import re
import csv

# --- For Azure ML Sandpit environment ---

# Project root path for Azure Sandpit environment
project_root_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone" 

# Change the current working directory to the project root
os.chdir(project_root_path)


# --- FIX 2: Bypass find_dotenv() and use a direct, verified path ---
dotenv_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env"

# Add a critical check to ensure the .env file exists at this path
if not os.path.exists(dotenv_path):
    raise FileNotFoundError(
        f"CRITICAL ERROR: .env file NOT FOUND at the expected path: {dotenv_path}\n"
        f"Please double-check the path you pasted into 'project_root_path'."
    )


# Load the .env file from the explicit, verified path
load_dotenv(dotenv_path=dotenv_path)

# The project root is now simply the current working directory
project_root = os.getcwd()

# --- Now, the rest of your variable loading will work correctly ---
relative_data_dir = os.getenv("VECTOR_DATASET_DIR")

# Add a check to make sure the variable was loaded successfully from the file
if not relative_data_dir:
    raise ValueError(
        "ERROR: 'VECTOR_DATASET_DIR' was not found in your .env file, or the file is empty."
    )

data_directory = os.path.join(project_root, relative_data_dir)

hf_token = os.getenv("HUGGINGFACE_TOKEN")
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# --- Final Verification ---
print(f"✅ Project root successfully set to: {project_root}")
print(f"✅ .env file loaded from: {dotenv_path}")
print(f"📁 Data directory set to: {data_directory}")

2025-08-24 15:46:16.484275: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-24 15:46:16.484373: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-24 15:46:17.111022: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-24 15:46:18.288288: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-24 15:46:23.261356: W tensorflow/compiler/tf2

✅ Project root successfully set to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/hj-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone
✅ .env file loaded from: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env
📁 Data directory set to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/hj-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Vector_Dataset


## Vector/Graph Store Data Ingestion

In [2]:
# Check if directory exists
if not data_directory or not os.path.isdir(data_directory):
    raise ValueError(
        f"The path '{data_directory}' is not a valid directory. "
        "Please check that the VECTOR_DATASET_DIR variable is set correctly in your .env file "
        "and that the directory actually exists."
    )

# Initialize LlamaParse with your API key
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
if not llama_cloud_api_key:
    raise ValueError("LLAMA_CLOUD_API_KEY not found in your .env file. Please get a key from https://cloud.llamaindex.ai")

parser = LlamaParse(
    api_key=llama_cloud_api_key,
    result_type="markdown",
    verbose=True
)

# Separate the file paths based on their type (PDF vs. other)
pdf_filepaths = []
other_filepaths = []
for filename in os.listdir(data_directory):
    file_path = os.path.join(data_directory, filename)
    if os.path.isfile(file_path):
        if filename.lower().endswith('.pdf'):
            pdf_filepaths.append(file_path)
        else:
            other_filepaths.append(file_path)

print(f"--- Found {len(pdf_filepaths)} PDF(s) and {len(other_filepaths)} other file(s) to process. ---")

# Process the files in batches
all_documents = []

# Process all PDFs in a single batch call to LlamaParse
if pdf_filepaths:
    print("\n- Parsing PDF files with LlamaParse...")
    try:
        # Calling parser.load_data() with a LIST of files is the correct way
        pdf_docs = parser.load_data(pdf_filepaths)
        all_documents.extend(pdf_docs)
        print(f"  -> Successfully parsed {len(pdf_filepaths)} PDF file(s).")
    except Exception as e:
        print(f"  -> FAILED to parse PDFs with LlamaParse. Error: {e}")

# Process all other files in a single batch call to SimpleDirectoryReader
if other_filepaths:
    print("\n- Parsing other files with SimpleDirectoryReader...")
    try:
        other_docs = SimpleDirectoryReader(input_files=other_filepaths).load_data()
        all_documents.extend(other_docs)
        print(f"  -> Successfully parsed {len(other_filepaths)} other file(s).")
    except Exception as e:
        print(f"  -> FAILED to parse other files. Error: {e}")

# The 'documents' variable should now contain all chunks from all parsed files
documents = all_documents
print(f"\n--- Ingestion complete ---")
print(f"Successfully loaded and chunked a total of {len(documents)} document(s) from all files in '{data_directory}'.")


--- Found 8 PDF(s) and 0 other file(s) to process. ---

- Parsing PDF files with LlamaParse...


Parsing files:   0%|          | 0/8 [00:00<?, ?it/s]

Started parsing the file under job_id 9b481df1-116e-41a2-94f6-0cd778c015a2
Started parsing the file under job_id 18b6bd05-dc84-40ac-b673-eadfd38f869f
Started parsing the file under job_id 6442ead3-bdaa-4cde-9802-851adec538b3
Started parsing the file under job_id 26d93173-27e4-4efc-8392-11d4a9c8d1be


Parsing files:  50%|█████     | 4/8 [00:08<00:04,  1.21s/it]

Started parsing the file under job_id 3a815bc1-a167-44da-a914-a164fec0c1ba
Started parsing the file under job_id b7beb2a2-5ae7-472b-ac2e-5b03d04b9000
Started parsing the file under job_id 1423e0fb-590e-44a7-8389-0e8506f9dc4d
Started parsing the file under job_id 45129857-1be9-4f92-9011-43a1125905dd


Parsing files: 100%|██████████| 8/8 [00:51<00:00,  6.44s/it]

  -> Successfully parsed 8 PDF file(s).

--- Ingestion complete ---
Successfully loaded and chunked a total of 137 document(s) from all files in '/mnt/batch/tasks/shared/LS_root/mounts/clusters/hj-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Vector_Dataset'.


## Llama 3.1 8B Instruct

In [ ]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)

llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    device_map="auto",
    max_new_tokens=1024,  # Set max_new_tokens here to avoid conflicts
    model_kwargs={"token": hf_token, "torch_dtype": torch.bfloat16},
    generate_kwargs={
        "temperature": 0.1,
        "do_sample": True,
        # Removed eos_token_id for more natural complete sentences
    }
)

print("HuggingFaceLLM initialized for conversational responses.")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

HuggingFaceLLM initialized for conversational responses.


## Vector Embeddings

In [4]:
Settings.llm = llm
Settings.embed_model = "local:BAAI/bge-small-en-v1.5"

print("Global settings configured with Llama 3.1 and bge-small embedding model.")

Global settings configured with Llama 3.1 and bge-small embedding model.


In [ ]:
# Pass the list 'documents' directly, creating separate index entries for each document chunk
# By default, VectorStoreIndex chunk size is set to 1024 characters
# And chunk overlap is set to 20% of the chunk size
# Each chunk will be 1024 characters long with a 200 character overlap
node_parser = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=200
)

pipeline = IngestionPipeline(
    transformations=[node_parser]
)

nodes = pipeline.run(documents=documents)

index = VectorStoreIndex(nodes)

print(f"Vector store index has been built successfully from {len(documents)} source document(s).")
print(f"Total nodes created with custom chunking: {len(nodes)}")
print(f"Chunk size: {node_parser.chunk_size}, Chunk overlap: {node_parser.chunk_overlap}")


Vector store index has been built successfully from 137 source document(s).
Total nodes created with custom chunking: 137
Chunk size: 1024, Chunk overlap: 200


## Retrieve Answer from Datastore

In [6]:
query_text_vector = "Which scam type had the highest number of cases in 2020?"

# Retriever to get the top 3 most similar nodes from the index
retriever = index.as_retriever(similarity_top_k=3)
retrieved_nodes = retriever.retrieve(query_text_vector)

raw_chunks = [n.get_content() for n in retrieved_nodes]

seen = set()
cleaned_chunks = []

for txt in raw_chunks:
    # Strip known cues that cause echoing
    t = txt.replace("(1 sentence)", "").strip()

    # De-duplicate identical chunks
    if t and t not in seen:
        cleaned_chunks.append(t)
        seen.add(t)

if not cleaned_chunks:
    cleaned_chunks = [txt.replace("(1 sentence)", "").strip() for txt in raw_chunks if txt.strip()]

# Combine the content of the retrieved nodes into a single context string
# Contains the "exact answer" material for the LLM
exact_context = "\n\n---\n\n".join(cleaned_chunks)

conversational_prompt_template = PromptTemplate(
    "You are a precise Q&A assistant. Use ONLY the context to answer the single question. "
    "Do not invent or add anything not present in the context. Do not repeat the question. "
    "Do not answer any other questions. Output exactly one sentence and then stop.\n\n"
    "Context:\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n\n"
    "User's Question: {query_str}\n"
    "Answer: "
)

# Format the prompt with our retrieved context and the original query
final_prompt = conversational_prompt_template.format(
    context_str=exact_context,
    query_str=query_text_vector
)

# Generate with a tight token cap
raw_response = llm.complete(
    final_prompt,
    max_new_tokens=64
)

# Trim to first sentence to prevent duplication or rambling
def first_sentence(text: str) -> str:
    s = text.strip()
    # Simple split on period
    parts = s.split(".")
    return (parts[0] + ".").strip() if parts and parts else s

extracted_answer = first_sentence(str(raw_response))
print(extracted_answer)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


E-commerce scams.


## Transform Extracted Answer To Be Conversational

In [7]:
# Normalize the extracted answer
# Eg "19,966." -> "19,966"
answer_core = extracted_answer.strip()
answer_core = re.sub(r"\.\s*$", "", answer_core).strip()

if not answer_core:
    print("Sorry, I couldn’t extract an answer from the context.")
else:
    REPHRASE_PROMPT = (
        "You are a precise assistant. Write exactly ONE conversational sentence that answers the question.\n"
        "Hard constraints:\n"
        "- Use the Extracted answer exactly once.\n"
        "- Do not add any other information not present in the Extracted answer or the Question.\n"
        "- Do not repeat yourself.\n"
        "- End with a single period.\n\n"
        f"Extracted answer: {answer_core}\n"
        f"Question: {query_text_vector}\n"
        "Answer:"
    )

    # Generate a short rephrased sentence to avoid loops
    rephrase_raw = llm.complete(REPHRASE_PROMPT, max_new_tokens=32)

    # Collapse whitespace
    s = " ".join(str(rephrase_raw).strip().split())

    # Remove a potential leading echo of the bare answer (e.g., "19,966. The total ...")
    if answer_core:
        s = re.sub(rf"^\s*{re.escape(answer_core)}\.\s*", "", s).strip()

    # Ensure exactly one sentence
    idx = s.find(".")
    s = (s[: idx + 1] if idx != -1 else s + ".").strip()

    # Enforce inclusion of the extracted answer exactly once
    # If the model dropped the value, fall back to the minimal guaranteed version
    if answer_core not in s:
        s = f"{answer_core}."

    print(s)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


E-commerce scams had the highest number of cases in 2020.


## Benchmarking

In [11]:
# --- Config paths ---
relative_benchmark_path = os.getenv("VECTOR_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("VECTOR_BENCHMARK_DATASET_DIR not set in .env")
BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "(test)vector_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Load and Prepare Data ---
if not os.path.isfile(BENCHMARK_FILE_PATH):
    raise FileNotFoundError(f"Benchmark file not found at: {BENCHMARK_FILE_PATH}")

# Load the entire benchmark CSV for processing
benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)

# --- CONFIG: Set the number of rows to test ---
# Slice to first 3 rows for a quick test, or comment out to run on the full file.
benchmark_df = benchmark_df.head(3)
print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")


# --- Unified Generation and Context Pinpointing Logic ---
from sentence_transformers import SentenceTransformer, util

# Load a sentence transformer model to find the most relevant context
# This should match your embedding model for consistency
similarity_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

def get_response_and_pinpoint_context(question: str, retriever):
    """
    Retrieves context, generates a response, and identifies the single most relevant 
    context chunk that was used for the answer.
    """
    # 1. Retrieve nodes
    retrieved_nodes = retriever.retrieve(question)
    
    # De-duplicate and clean chunks
    seen = set()
    cleaned_chunks = []
    for node in retrieved_nodes:
        content = node.get_content().replace("(1 sentence)", "").strip()
        if content and content not in seen:
            cleaned_chunks.append(content)
            seen.add(content)
            
    # Combine into a single context string for the LLM
    exact_context_for_llm = "\n\n---\n\n".join(cleaned_chunks)

    # 2. Use the same successful prompt template
    prompt_template = PromptTemplate(
        "You are a precise Q&A assistant. Use ONLY the context to answer the single question. "
        "Do not invent or add anything not present in the context. Do not repeat the question. "
        "Do not answer any other questions. Output exactly one sentence and then stop.\n\n"
        "Context:\n"
        "---------------------\n"
        "{context_str}\n"
        "---------------------\n\n"
        "User's Question: {query_str}\n"
        "Answer: "
    )
    final_prompt = prompt_template.format(context_str=exact_context_for_llm, query_str=question)
    
    # 3. Generate response
    raw_response = llm.complete(final_prompt, max_new_tokens=64)
    
    # Clean the response
    s = str(raw_response).strip()
    parts = s.split(".")
    prediction = (parts[0] + ".").strip() if parts else s
    
    # 4. Pinpoint the most relevant context chunk
    most_relevant_chunk = ""
    if prediction and cleaned_chunks:
        # Encode the generated answer and all context chunks
        answer_embedding = similarity_model.encode(prediction, convert_to_tensor=True)
        chunk_embeddings = similarity_model.encode(cleaned_chunks, convert_to_tensor=True)
        
        # Compute cosine similarities
        cosine_scores = util.cos_sim(answer_embedding, chunk_embeddings)
        
        # Find the chunk with the highest score
        best_chunk_index = cosine_scores.argmax()
        most_relevant_chunk = cleaned_chunks[best_chunk_index]
    
    return prediction, cleaned_chunks, most_relevant_chunk


# --- Generate Predictions ---
ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
retriever = index.as_retriever(similarity_top_k=3) # Using top 3 as before

print("\nGenerating predictions and pinpointing context for each question...")
# Initialize new columns to store results
benchmark_df['response'] = ''
benchmark_df['retrieved_contexts'] = ''

for i, row in benchmark_df.iterrows():
    question = row['Question'].strip()
    ground_truth = row['Answer'].strip()
    
    # Get response and the single best context chunk
    prediction, all_retrieved_chunks, single_best_chunk = get_response_and_pinpoint_context(question, retriever)
    
    # Store results in the DataFrame
    benchmark_df.at[i, 'response'] = prediction
    # Store ONLY the single best chunk for clarity
    benchmark_df.at[i, 'retrieved_contexts'] = single_best_chunk
    
    # Collect data for Ragas evaluation (Ragas still needs all chunks)
    ragas_data["question"].append(question)
    ragas_data["answer"].append(prediction)
    ragas_data["contexts"].append(all_retrieved_chunks)
    ragas_data["ground_truth"].append(ground_truth)
    
    print(f"Processed {i+1}/{len(benchmark_df)}")

# --- Run Ragas Evaluation ---
ragas_dataset = Dataset.from_dict(ragas_data)
print(f"\nPrepared {len(ragas_dataset)} valid samples for Ragas evaluation.")

judge_llm = llm_factory(model="gpt-4o") 
judge_embeddings = embedding_factory(model="text-embedding-ada-002")

# Define metrics
from ragas.metrics import answer_relevancy, faithfulness, context_recall, context_precision

metrics_to_evaluate = [
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision
]

print("\nRunning Ragas evaluation...")
result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics_to_evaluate,
    llm=judge_llm,
    embeddings=judge_embeddings,
)
print("Ragas evaluation complete.")

# --- Format and Save Final Results ---
scores_df = result.to_pandas()

# Merge scores back into the main DataFrame
benchmark_df = benchmark_df.reset_index(drop=True)
scores_df = scores_df.reset_index(drop=True)
metric_cols = [m.name for m in metrics_to_evaluate]
benchmark_df = benchmark_df.join(scores_df[metric_cols])

# Ensure the desired column order
final_columns = ['Question', 'Answer', 'Context', 'response', 'retrieved_contexts'] + metric_cols
benchmark_df = benchmark_df[final_columns]

# Save the final results to a CSV file
benchmark_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\nDetailed results saved to: {OUTPUT_FILE_PATH}")

# Print overall performance metrics
print("\n--- Overall Ragas Performance Metrics (Averages) ---")
print(scores_df[metric_cols].mean(numeric_only=True))
print("----------------------------------------------------")


Loaded 3 question-answer pairs for evaluation.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Processed 1/3


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed 2/3
Processed 3/3

Prepared 3 valid samples for Ragas evaluation.

Running Ragas evaluation...


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

Ragas evaluation complete.

Detailed results saved to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/hj-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/(test)vector_benchmark_results.csv

--- Overall Ragas Performance Metrics (Averages) ---
answer_relevancy     0.583556
faithfulness         1.000000
context_recall       1.000000
context_precision    1.000000
dtype: float64
----------------------------------------------------
